# Phase-side look at Umbra ships — frequency response and sub-aperture coherence

The question this notebook answers: **at 25 cm, do man-made targets separate
from clutter in the phase domain?** If they do, a phase-side detector is worth
building. If they don't, you've found out in an afternoon.

1. install + upload `spectrum_view.py`
2. browse Umbra's ship collection and download one scene
3. check whether the scene's JSON carries bounding boxes
4. get boxes: from the JSON, from a simple auto-finder, or typed by hand
5. frequency response (k-space) panels
6. **the actual test** — sub-aperture coherence inside boxes vs clutter, with a
   separability number rather than an eyeball
7. sweep the number of looks and the chip size

## 1. Install and upload

In [ ]:
!pip -q install "sarpy==2.1.1" boto3 numpy scipy matplotlib
from google.colab import files
up = files.upload()          # choose spectrum_view.py
import importlib.util, sys
spec = importlib.util.spec_from_file_location("sv", "spectrum_view.py")
sv = importlib.util.module_from_spec(spec); sys.modules["sv"] = spec.loader.exec_module(sv) or sv
print("loaded spectrum_view")

## 2. Umbra's ship collection

`sar-data/tasks/ship_detection_testdata/` is ~1000 scenes of ships worldwide
(~7.4 TB in all formats). We list only SICD files and their JSON metadata.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "umbra-open-data-catalog"
PREFIX = "sar-data/tasks/ship_detection_testdata/"
s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))

sicds, jsons = [], []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for o in page.get("Contents", []):
        k = o["Key"]
        if k.endswith("_SICD.nitf"): sicds.append((o["Size"], k))
        elif k.endswith(".json"):    jsons.append((o["Size"], k))
    if len(sicds) > 40: break

sicds.sort()
print(f"{len(sicds)} SICDs found (smallest first), {len(jsons)} json files\n")
for i,(sz,k) in enumerate(sicds[:15]):
    print(f"[{i:2d}] {sz/1e6:7.1f} MB  {k.split('/')[-1]}")
if not sicds:
    print("no SICD under this prefix — this collection may only carry GEC/TIF;"
          " fall back to the general catalogue: sar-data/tasks/")

## 3. Download one scene (SICD + every JSON beside it)

In [ ]:
import os, json
CHOICE = 0
size, key = sicds[CHOICE]
folder = key.rsplit("/", 1)[0]
local = "/content/" + key.rsplit("/", 1)[-1]
if not os.path.exists(local):
    print(f"downloading {size/1e6:.1f} MB ...")
    s3.download_file(BUCKET, key, local)
print("SICD:", local)

meta_local = []
for sz, jk in jsons:
    if jk.rsplit("/", 1)[0] == folder:
        p = "/content/" + jk.rsplit("/", 1)[-1]
        s3.download_file(BUCKET, jk, p); meta_local.append(p)
print("json files:", meta_local)

## 4. Does the JSON carry boxes?

Umbra's metadata is collection information (geometry, timing). Published
write-ups on this collection ran their own detector to get boxes, which
suggests ground truth is **not** included — but check rather than assume.

In [ ]:
import json
def find_key(obj, want, path=""):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if want in k.lower(): hits.append((path + "/" + k, v))
            hits += find_key(v, want, path + "/" + k)
    elif isinstance(obj, list) and obj:
        hits += find_key(obj[0], want, path + "/[0]")
    return hits

for p in meta_local:
    d = json.load(open(p))
    print("==", p, "top-level keys:", list(d)[:12])
    for want in ("box", "annot", "label", "detect", "ship", "target"):
        for path, v in find_key(d, want):
            print(f"   {want}: {path} -> {str(v)[:120]}")

## 5. Load a chip and (if needed) get boxes

Three routes, in order of preference:
* `BOXES_FROM_JSON` if section 4 found real annotations,
* the auto-finder below — a crude bright-blob detector, fine for *locating*
  candidates you then eyeball,
* typing coordinates by hand after looking at the image.

The auto-finder is not a detector and its output is not ground truth. It just
saves you hunting for ships in a 10k-pixel image.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

CHIP_SIZE = 1024          # try 256 / 512 / 1024
img, m = sv.load_sicd_chip(local, None, CHIP_SIZE)
print("chip", m["chip"], "of", m["full"])
print(f"range res {0.886/m['row_irbw']:.2f} m, azimuth res {0.886/m['col_irbw']:.2f} m, "
      f"tapers {m['row_wgt_type']}/{m['col_wgt_type']}")

plt.figure(figsize=(9,9))
plt.imshow(sv.db(img), cmap="gray"); plt.title("chip — find your ships here"); plt.colorbar(fraction=0.046)
plt.grid(alpha=0.2, color="c"); plt.show()

In [ ]:
from scipy import ndimage

def auto_boxes(img, k=6.0, min_pix=30, pad=4):
    """Bright-blob finder: threshold at mean + k*sigma of |I|, label, keep
    blobs above min_pix. Crude on purpose — candidates, not detections."""
    a = np.abs(img).astype(float)
    sm = ndimage.uniform_filter(a, 3)
    thr = sm.mean() + k * sm.std()
    lab, n = ndimage.label(sm > thr)
    out = []
    for i, sl in enumerate(ndimage.find_objects(lab), 1):
        if (lab[sl] == i).sum() < min_pix: continue
        r0, r1 = max(0, sl[0].start-pad), min(img.shape[0], sl[0].stop+pad)
        c0, c1 = max(0, sl[1].start-pad), min(img.shape[1], sl[1].stop+pad)
        out.append({"name": f"cand{len(out)}", "box": [r0, c0, r1, c1]})
    return out

cands = auto_boxes(img)
print(f"{len(cands)} candidates")

# chip-relative -> the format spectrum_view expects (full-image pixels)
r0c, c0c, _, _ = m["chip"]
BOXES = [{"name": b["name"],
          "box": [b["box"][0]+r0c, b["box"][1]+c0c, b["box"][2]+r0c, b["box"][3]+c0c]}
         for b in cands]
# ---- or type your own, in FULL-image pixels:
# BOXES = [{"name":"ship1","box":[4200, 5100, 4260, 5190]}]

import json; json.dump(BOXES, open("/content/boxes.json","w")); print(BOXES[:5])

In [ ]:
from matplotlib.patches import Rectangle
boxes = sv.load_boxes("/content/boxes.json", m["chip"])
fig, ax = plt.subplots(figsize=(9,9))
ax.imshow(sv.db(img), cmap="gray")
for name, r0, c0, r1, c1 in boxes:
    ax.add_patch(Rectangle((c0,r0), c1-c0, r1-r0, fill=False, ec="lime", lw=1.2))
    ax.text(c0, r0-3, name, color="lime", fontsize=7)
ax.set_title(f"{len(boxes)} boxes on the chip"); plt.show()

## 6. Frequency response panels

In [ ]:
!python spectrum_view.py "{local}" --chip-size {CHIP_SIZE} --boxes /content/boxes.json --looks 3 -o /content/spec.png
from IPython.display import Image, display
display(Image("/content/spec.png"))

## 7. The actual test — does coherence separate targets from clutter?

Box-median coherence is a weak statistic: a ship is bright scatterers *plus*
shadow and gaps, so the median gets dragged down by the dark pixels inside the
box. Better summaries:

* **p90 inside the box** — the coherent scatterers, not the gaps
* **AUC** — probability a random in-box pixel scores above a random clutter
  pixel. 0.5 = no separation, 1.0 = perfect. This is the number to report.

In [ ]:
def coherence_stats(img, boxes, looks=3, win=7):
    L = sv.sub_aperture(img, looks)
    gam = np.mean([sv.coherence(L[i], L[i+1]) for i in range(len(L)-1)], axis=0)
    inside = np.zeros(gam.shape, bool)
    for _, r0, c0, r1, c1 in boxes: inside[r0:r1, c0:c1] = True
    a, b = gam[inside], gam[~inside]
    if a.size == 0: return gam, None
    # AUC via rank statistic (Mann-Whitney), subsampled for speed
    rs = np.random.default_rng(0)
    bs = b if b.size < 200000 else rs.choice(b, 200000, replace=False)
    allv = np.concatenate([a, bs]); order = allv.argsort().argsort() + 1
    ra = order[:a.size].sum()
    auc = (ra - a.size*(a.size+1)/2) / (a.size * bs.size)
    return gam, {"n_target": int(a.size), "n_clutter": int(b.size),
                 "median_target": float(np.median(a)), "median_clutter": float(np.median(b)),
                 "p90_target": float(np.percentile(a,90)), "p90_clutter": float(np.percentile(b,90)),
                 "auc": float(auc)}

gam, st = coherence_stats(img, boxes, looks=3)
print(st)

fig, ax = plt.subplots(1, 2, figsize=(14,5.5))
im = ax[0].imshow(gam, cmap="viridis", vmin=0, vmax=1); ax[0].set_title("sub-aperture coherence, 3 looks")
for name, r0, c0, r1, c1 in boxes:
    ax[0].add_patch(Rectangle((c0,r0), c1-c0, r1-r0, fill=False, ec="red", lw=1.2))
plt.colorbar(im, ax=ax[0], fraction=0.046)
if st:
    inside = np.zeros(gam.shape, bool)
    for _, r0, c0, r1, c1 in boxes: inside[r0:r1, c0:c1] = True
    ax[1].hist(gam[~inside].ravel(), bins=80, density=True, alpha=.6, label="clutter")
    ax[1].hist(gam[inside].ravel(),  bins=80, density=True, alpha=.6, label="in box")
    ax[1].set_xlabel("coherence"); ax[1].set_ylabel("density")
    ax[1].set_title(f"AUC = {st['auc']:.3f}   (0.5 = no separation)"); ax[1].legend()
plt.tight_layout(); plt.show()

### How to read the AUC

* **> 0.8** — coherence is a real discriminator at this resolution. Worth
  building the channel-stacking detector (`[log|I|, coherence, sub-band coherence]`
  into a pretrained backbone).
* **0.6 – 0.8** — weak but present; useful as an extra channel, not alone.
* **≈ 0.5** — no separation. The phase-side story doesn't hold at 25 cm for
  these targets, and that is a genuinely useful negative result.

Caveat: if your boxes came from the auto-finder, they are *bright* blobs by
construction, so the AUC is optimistic — brightness and coherence correlate.
Hand-drawn or SDFSD boxes give the honest number.

## 8. Sweeps — number of looks, and chip size

In [ ]:
import pandas as pd
rows = []
for looks in (2, 3, 4, 6, 8):
    _, st = coherence_stats(img, boxes, looks=looks)
    if st: rows.append({"looks": looks, **{k: round(v,3) for k,v in st.items() if k!="n_clutter"}})
pd.DataFrame(rows)

In [ ]:
rows = []
for size in (256, 512, 1024):
    im2, m2 = sv.load_sicd_chip(local, None, size)
    bx = sv.load_boxes("/content/boxes.json", m2["chip"])
    if not bx:
        rows.append({"chip": size, "note": "no boxes inside this chip"}); continue
    _, st = coherence_stats(im2, bx, looks=3)
    rows.append({"chip": size, "boxes": len(bx), "auc": round(st["auc"],3),
                 "p90_target": round(st["p90_target"],3), "p90_clutter": round(st["p90_clutter"],3)})
pd.DataFrame(rows)

## Notes

* Smaller chips centre on the scene centre pixel, so boxes near the edge drop
  out — that's why the chip sweep reports how many boxes survived.
* More looks = finer aspect sampling but lower SNR per look, so coherence
  falls. The useful number of looks is where AUC peaks, not where coherence is
  highest.
* If Umbra's JSON has no boxes, **SDFSD-v1.0** has rotated-box shapefiles over
  211 Umbra spotlight scenes and is the right source for labelled work.
* Everything here is amplitude-and-coherence on the SICD. No CPHD needed —
  this is the cheap version of the phase-side question, and it should be
  answered before any signal-domain training.